# 18 · Deployment & Serving — Predictions as a Service

**Part 2 · Machine Learning Systems** · *Prerequisites: 01, 17* · *Book mapping: DMLS ch. 7*

A model in a notebook is a promise; a model behind an endpoint is a product. Deployment is
where ML rejoins Part 1 on Part 1's terms: latency budgets (nb 01), queueing (nb 01),
caching layouts (nb 03), and the freshness trades of derived data (nb 12) — because a
prediction *is* derived data, and the core serving decision is the same one: **compute
ahead of time, or on demand?**

**Learning objectives.**

1. Quantify the **batch vs online prediction** trade: waste + staleness vs latency +
   infrastructure.
2. Measure why **batching** requests is the fundamental serving optimization
   (vectorization), then simulate a **dynamic batcher** and its latency/throughput dial.
3. Compress models three ways — **quantization** (measured 8×), **pruning** (measured on
   a boosted ensemble), **distillation** (built) — and know what each trades.
4. Reason about cloud vs edge with nb 01's latency ladder.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=18)

## 1. Batch vs online prediction

Two ends of a dial:

- **Batch prediction**: a nightly job scores *every* user; serving is a key-value lookup
  (nb 03's fastest read). Simple, cheap per request, latency-proof — and predictions are
  up to a day **stale**, computed for many users who never show up.
- **Online prediction**: compute at request time. Always fresh, compute exactly what's
  needed — and now the model's latency sits inside the user's click, with all of nb 01's
  queueing consequences.

Put numbers on the trade for a 50k-user product where ~10% visit daily and user state
changes intraday for some:

In [ ]:
N_USERS, DAILY_ACTIVE, INTRADAY_CHANGE = 50_000, 0.10, 0.08

active = rng.random(N_USERS) < DAILY_ACTIVE
state_changed_since_batch = rng.random(N_USERS) < INTRADAY_CHANGE

batch_compute = N_USERS                                   # score everyone nightly
batch_wasted = int((~active).sum())                       # scored, never requested
batch_stale_served = int((active & state_changed_since_batch).sum())
online_compute = int(active.sum())                        # score only who shows up

print(f"batch:  {batch_compute:,} scores/night, {batch_wasted/batch_compute:.0%} wasted, "
      f"{batch_stale_served:,} users served a STALE prediction")
print(f"online: {online_compute:,} scores/day (right-sized), 0 stale - "
      f"but every request now waits for the model")
assert batch_wasted / batch_compute == (~active).mean()
assert batch_stale_served > 300, "staleness is not hypothetical at these rates"
assert online_compute < batch_compute / 5, "online computes an order of magnitude less"

Neither wins in general — the decision keys on **how fast the input changes relative to
how often it's used**: churn scores refreshed weekly? Batch. Fraud scores on a
transaction that didn't exist a second ago? Online, no choice. Recommendations? Usually
hybrid: batch-compute candidate sets (capstone 24), online-rank with fresh context. And
freshness has *two* dials — prediction freshness and **feature** freshness (nb 16's
online store) — giving the classic 2×2 every serving design fills in (exercise 5).

## 2. The physics of serving: batching

The single most important serving fact: models score a **batch** of inputs far cheaper
than the same inputs one at a time — vectorized math amortizes per-call overhead (for
GPUs, multiply this effect by orders of magnitude). Measure it:

In [ ]:
# A representative tabular model:
X_train = rng.normal(0, 1, (10_000, 8))
y_train = (X_train[:, 0] * X_train[:, 1] + X_train[:, 2] > 0.2).astype(int)
model = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)
X_requests = rng.normal(0, 1, (256, 8))

t0 = time.perf_counter()
for i in range(256):
    model.predict_proba(X_requests[i:i + 1])              # 256 lonely requests
t_single = time.perf_counter() - t0
t0 = time.perf_counter()
model.predict_proba(X_requests)                           # one batch of 256
t_batch = time.perf_counter() - t0

per_row_single = t_single / 256 * 1e3
per_row_batch = t_batch / 256 * 1e3
print(f"per-prediction cost - one at a time: {per_row_single:.3f} ms, "
      f"batched x256: {per_row_batch:.4f} ms  ({per_row_single/per_row_batch:.0f}x)")
assert t_single > 5 * t_batch, "vectorization is the free lunch serving is built on"

But requests arrive one at a time. A **dynamic batcher** holds each request up to a small
window `W`, then scores whatever accumulated as one batch — deliberately *adding* a little
latency to buy a lot of throughput. Simulate it with the costs we just measured:

In [ ]:
overhead_ms = per_row_single - per_row_batch     # fixed cost per model call
marginal_ms = per_row_batch                      # extra cost per row in a call

# Load the server at 97% of its PER-REQUEST capacity (nb 01's danger zone),
# defined from the service time we just measured, so the experiment is about
# queueing physics rather than this machine's speed:
rate_per_ms = 0.97 / per_row_single

def serve_day(window_ms: float, batching: bool, n_requests: int = 30_000, seed: int = 0):
    """One server, Poisson arrivals. batching=False -> one request per model
    call; batching=True -> each call waits `window_ms`, then scores everything
    that has arrived (including any backlog). Batch cost: overhead + B*marginal."""
    r = np.random.default_rng(seed)
    arrivals = np.cumsum(r.exponential(1 / rate_per_ms, n_requests))
    latencies, i, server_free = [], 0, 0.0
    while i < n_requests:
        start = max(arrivals[i], server_free)     # when the server takes the head
        if batching:
            horizon = start + window_ms
            j = int(np.searchsorted(arrivals, horizon, side="right"))
            idx = np.arange(i, max(j, i + 1))     # the window's harvest (>= 1)
            done = horizon + overhead_ms + len(idx) * marginal_ms
        else:
            idx = np.array([i])
            done = start + overhead_ms + marginal_ms
        latencies.extend(done - arrivals[idx])
        server_free = done
        i = int(idx[-1]) + 1
    return np.percentile(latencies, [50, 99])

rows = []
for mode, w in [("per-request", 0.0), ("batched", per_row_single),
                ("batched", 3 * per_row_single), ("batched", 8 * per_row_single)]:
    p50, p99 = serve_day(w, batching=(mode == "batched"))
    rows.append({"mode": mode, "window_ms": round(w, 2),
                 "p50_ms": round(float(p50), 1), "p99_ms": round(float(p99), 1)})
table = pd.DataFrame(rows)
print(f"arrival rate: {rate_per_ms:.2f} req/ms  (97% of per-request capacity)")
print(table.to_string(index=False))

per_request_p99 = table.loc[table["mode"] == "per-request", "p99_ms"].item()
batched_p99 = table["p99_ms"].iloc[2]
assert per_request_p99 > 5 * batched_p99, \
    "at 97% utilization, per-request serving lives on nb 01's hockey stick - " \
    "a small batching window buys the p99 back many times over"

Read the table with nb 01 eyes: per-request mode runs at ρ=0.97, so requests spend most
of their "latency" *queueing* — the hockey stick, live. The batcher converts backlog
into batch size: a window of a few service-times harvests several requests per call,
multiplying capacity (per-row cost collapses toward `marginal`) and pulling the p99 down
by an order of magnitude — *including* the window it deliberately added. The window is a
dial: real-time products run tiny windows with more replicas; throughput products crank
it up. (On GPUs the overhead:marginal ratio is far more extreme, which is why GPU
serving without batching is setting money on fire.)

## 3. Model compression

Three levers, usually stacked, for when the model must be smaller or faster (edge
deployment, latency budgets, cost):

### 3.1 Quantization — fewer bits per number

Store weights as int8 + a scale factor instead of float64. For inference-only use, the
precision was mostly decoration:

In [ ]:
# A wide linear model (2,000 text-ish features) - where weight bytes add up:
V = 2_000
X_wide = (rng.random((6_000, V)) < 0.02).astype(np.float64)
w_true = rng.normal(0, 1, V) * (rng.random(V) < 0.1)
y_wide = ((X_wide @ w_true + rng.normal(0, 0.4, 6_000)) > 0).astype(int)
wide = LogisticRegression(max_iter=2000).fit(X_wide[:4000], y_wide[:4000])

w = wide.coef_[0]
scale = np.abs(w).max() / 127.0
w_int8 = np.round(w / scale).astype(np.int8)              # 8 bits per weight
w_dequant = w_int8.astype(np.float64) * scale

logits_fp = X_wide[4000:] @ w + wide.intercept_[0]
logits_q = X_wide[4000:] @ w_dequant + wide.intercept_[0]
auc_fp = roc_auc_score(y_wide[4000:], logits_fp)
auc_q = roc_auc_score(y_wide[4000:], logits_q)

print(f"weights: {w.nbytes:,} B float64 -> {w_int8.nbytes:,} B int8 "
      f"({w.nbytes / w_int8.nbytes:.0f}x smaller); AUC {auc_fp:.4f} -> {auc_q:.4f}")
assert w.nbytes == 8 * w_int8.nbytes
assert auc_fp - auc_q < 0.005, "8x smaller, accuracy within noise"

(Real quantization also quantizes *activations*, where outliers make the naive
max-scaling above hurt — hence per-channel scales and calibration datasets; exercise 3.)

### 3.2 Pruning — fewer parts

For tree ensembles, "parts" are trees, and inference cost is linear in them. Boosting
front-loads its value — measure how much of the ensemble is dispensable:

In [ ]:
gb = GradientBoostingClassifier(n_estimators=300, random_state=0).fit(
    X_train[:6000], y_train[:6000])
X_val, y_val = X_train[6000:], y_train[6000:]

staged_auc = [roc_auc_score(y_val, p[:, 1])
              for p in gb.staged_predict_proba(X_val)]
full_auc = staged_auc[-1]
enough = int(np.argmax(np.array(staged_auc) >= full_auc - 0.005)) + 1

fig, ax = plt.subplots()
ax.plot(range(1, 301), staged_auc)
ax.axvline(enough, color="#c62828", linestyle="--",
           label=f"{enough} trees: within 0.005 of full")
ax.set_xlabel("trees used at inference")
ax.set_ylabel("validation AUC")
ax.set_title("Pruning a boosted ensemble: most trees buy almost nothing")
ax.legend()
plt.show()

print(f"{enough}/300 trees reach full-model quality - "
      f"{1 - enough/300:.0%} of inference cost prunable")
assert enough < 200, "the value is front-loaded; the tail of the ensemble is a luxury"
assert staged_auc[99] > full_auc - 0.02, "100 trees land within 0.02 AUC of all 300"

### 3.3 Distillation — a small student of a big teacher

Train a compact **student** to imitate the **teacher's outputs** on abundant unlabeled
data. The student inherits behavior the tiny labeled set alone couldn't teach it:

In [ ]:
X_unlabeled = rng.normal(0, 1, (20_000, 8))               # cheap: no labels needed
teacher_labels = model.predict(X_unlabeled)               # the teacher speaks

X_eval = rng.normal(0, 1, (8_000, 8))
y_eval = (X_eval[:, 0] * X_eval[:, 1] + X_eval[:, 2] > 0.2).astype(int)

def with_cross(X):
    """Give the linear student the one interaction column it needs to have any
    chance of imitating the teacher (nb 16's crossing, in service of compression)."""
    return np.c_[X, X[:, 0] * X[:, 1]]

student_small = LogisticRegression(max_iter=1000).fit(
    with_cross(X_train[:200]), y_train[:200])             # 200 real labels only
student_distilled = LogisticRegression(max_iter=1000).fit(
    with_cross(X_unlabeled), teacher_labels)              # 20k teacher opinions

acc_teacher = (model.predict(X_eval) == y_eval).mean()
acc_small = (student_small.predict(with_cross(X_eval)) == y_eval).mean()
acc_distilled = (student_distilled.predict(with_cross(X_eval)) == y_eval).mean()
print(f"teacher (GBM): {acc_teacher:.3f} | student on 200 labels: {acc_small:.3f} "
      f"| student distilled on 20k teacher-labeled rows: {acc_distilled:.3f}")
assert acc_distilled > acc_small, "the teacher's opinions are a data multiplier"
assert acc_distilled > acc_teacher - 0.05, "the student gets close, at a fraction of the cost"

The honest caveat, visible in the numbers: distillation transfers only what the student
**can represent** (we had to hand the linear student a crossed feature). Teacher-student
gaps are representation gaps, not training failures.

## 4. Where the model runs: cloud vs edge

Consult nb 01's ladder: a cloud round trip is 20–140 ms *before* any computation —
already the whole latency budget of keyboard suggestions, camera filters, AR, or
anything offline. **Edge deployment** (on-device) removes the network entirely and keeps
raw data local (a privacy and cost win), at the price of: the device's compute/memory
envelope (hence §3 — compression is the visa for edge), fleet heterogeneity (thousands of
device profiles), and a *much* harder update and monitoring story (nb 19–20 assume you
can see predictions; on-device you must design telemetry deliberately). Cloud keeps
control, observability, and big models; edge keeps latency, privacy, and offline. Hybrids
dominate: small on-device model for the instant path, cloud model for the heavy path —
the same "cache + authoritative store" shape as nb 03/12.

## 5. Exercises

1. **Choose the mode.** Batch, online, or hybrid — and why: (a) email spam filtering;
   (b) weekly churn-risk emails to account managers; (c) homepage recommendations for a
   10M-user shop; (d) card-transaction fraud scoring.
2. **Latency budget.** An online prediction must return in 150 ms p99. Decompose a
   realistic budget across: edge→region network, feature fetches (online store, nb 16),
   model inference, and the batcher window from §2 — and identify which item nb 01's
   tail-amplification math punishes if you fan out feature fetches to 8 shards.
3. **Quantization's enemy.** Our max-based int8 scaling worked because linear-model
   weights are well-behaved. Why do *activation outliers* in deep nets break naive
   quantization, and what do per-channel scales + a calibration set fix?
4. **The 2×2.** Fill the feature-freshness × prediction-freshness matrix with one product
   per cell (batch/batch, batch features + online prediction, streaming features + online
   prediction, and the odd cell — streaming features + batch predictions: what even fits
   there?).
5. **Rollback drill.** Online model v2 misbehaves 40 minutes after deploy. Walk the
   rollback: what must have been true about the *artifact*, the *feature pipeline
   version* (nb 16), and the *registry* (nb 17 ex. 5) for rollback to be one command? What
   extra step does a *batch* model rollback need?

### Solutions

**1.** (a) Online-ish (score on arrival — mail tolerates ~seconds, so a queue consumer,
nb 11, not a request-path service). (b) Batch — weekly cadence, tiny audience.
(c) Hybrid — nightly batch candidate generation + online ranking with session context
(capstone 24's architecture). (d) Online, hard requirement — the transaction is the
input; plus a streaming feature path (capstone 25).

**2.** Example budget: network 30 ms + feature fetches 25 ms + inference 20 ms +
batching window 5 ms + app logic 20 ms ≈ 100 ms, leaving 50 ms headroom for tails.
Fanning feature fetches to 8 shards makes the fetch cost `max` of 8 draws —
1−(1−p)⁸ tail amplification (nb 00 §3.2): a shard p99 of 25 ms becomes a ~18% chance per
request of blowing the fetch budget; mitigations: fewer shards per request, hedged
fetches, or degrade-to-default features on timeout (fallbacks are a *feature contract*,
nb 25).

**3.** One outlier activation stretches the max-based scale so that 99% of values
squeeze into a few int8 buckets — precision collapses where the mass lives. Per-channel
scales localize the damage to the outlier's channel; a calibration set chooses clipping
ranges (e.g., percentile-based) so the scale fits the *distribution*, sacrificing exact
outliers for resolution where it matters.

**4.** batch/batch: weekly churn emails. batch features + online prediction: homepage
ranking with day-old user embeddings. streaming + online: fraud (capstone 25).
streaming features + batch predictions: the odd one — continuously maintained aggregates
consumed by a periodic decision, e.g., hourly dynamic-pricing runs reading live demand
counters; rare because if you can stream features you usually want the decision fresh
too.

**5.** Artifact: immutable, versioned model binaries (registry holds v1) — never
"latest" tags. Feature pipeline: v1's *feature definitions* must still be deployable —
model and features roll back as a **unit** (the pairing recorded in the registry,
nb 17). Router: traffic split flips back v2→v1 (the nb 20 canary machinery running in
reverse). Batch extra step: rolled-back model must **re-score** the affected cohort (or
you serve v2's cached predictions for up to a full cycle — derived data must be
re-derived, nb 12).

## 6. Takeaways

- Predictions are derived data; serving is the compute-ahead vs compute-on-demand trade
  with nb 12's freshness economics: batch = cheap + stale + wasteful, online = fresh +
  latency-bound, hybrid = the usual answer.
- **Batching is the physics**: vectorization makes per-row cost collapse; the dynamic
  batcher's window is a latency-for-throughput dial, and under load the unbatched path
  is a queueing disaster, not a baseline.
- Compression: quantize (8× for near-free on weights), prune (ensembles front-load their
  value), distill (transfers what the student can represent). Together they are the
  ticket to edge — where latency, privacy, and offline live.
- Deploy models *with* their feature-pipeline versions, immutably, behind a router that
  can roll back in one motion — nb 20 builds the process that decides when.

**Further reading.** DMLS ch. 7; Hinton et al., *Distilling the Knowledge in a Neural
Network* (2015); Crankshaw et al., *Clipper: A Low-Latency Online Prediction Serving
System* (NSDI 2017) — dynamic batching, productionized.

**Next:** [19 · Shifts & monitoring](19-shifts-monitoring.ipynb) — the model is live.
The world now begins, quietly, to stop being the training set.